# M11 on Kaggle: second verifier (Mistral 7B Instruct) on a subset of seed 42's rows

The proposal runs Mistral 7B "for a smaller subset". This notebook takes the rows seed 42's Gemma run sent to the verifier, in a seeded random order drawn without labels, and verifies them with Mistral until a time budget (9 hours) runs out. No refit and no CICIoMT2024 input are needed: the evidence comes from that run. Results are rewritten after every 160 rows, so a timeout still leaves a result.

**Attach as input:**
1. **Seed 42's M11 output:** upload `m11_full_seed42_outputs.zip` (the one you brought back) as a new private Kaggle dataset and attach it. The notebook needs `evidence.parquet`, `verified.parquet` and `centroids.json` from it.
2. **The model:** *Add Input → Models →* **Mistral**, framework *Transformers*, an instruct 7B variation (v0.2 if offered; accept the licence on the model page first). Without it, the notebook downloads `mistralai/Mistral-7B-Instruct-v0.2` from Hugging Face, which needs a Kaggle secret `HF_TOKEN` from an account that accepted that model's terms.

**Settings:** Internet *On*. Accelerator: **GPU T4**. **Run:** *Save Version → Save & Run All (Commit)*.

**Output:** `/kaggle/working/m11_mistral_outputs.zip` with `reports/m11/full/mistral/` and the run folder. Bring it back, also after a failure.

In [ ]:
import os, subprocess, sys
REPO = "/kaggle/working/IDS-code"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Indrajit147/IDS-code.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
# --no-deps: keep Kaggle's preinstalled stack intact; versions are recorded per run. bitsandbytes isn't preinstalled.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO, "--no-deps"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"], check=True)
os.chdir(REPO)
sys.path.insert(0, f"{REPO}/src")   # this kernel started before the editable install, so it cannot see it
os.environ["IDS_REPO_ROOT"] = REPO
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
assert os.path.exists("scripts/m11_subset.py"), "This clone predates M11 (no scripts/m11_subset.py). Restart the session so it clones the latest main."

In [ ]:
# The model: a Kaggle model input if attached, else Hugging Face (needs the HF_TOKEN secret)
import glob
found = sorted(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/config.json", recursive=True)
               if "mistral" in p.lower() and "instruct" in p.lower() and "7b" in p.lower())
print("Mistral inputs found:", found)
if found:
    MODEL = ([f for f in found if "v0.2" in f.lower() or "v0-2" in f.lower()] or found)[-1]
else:
    MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("model:", MODEL)

In [ ]:
import json
# Seed 42's M11 output -> reports/m11/full/ (Kaggle may or may not have unzipped the upload)
import shutil, zipfile
NEED = ["evidence.parquet", "verified.parquet", "centroids.json", "summary.json"]
for z in glob.glob("/kaggle/input/**/m11_full_seed42_outputs*.zip", recursive=True):
    zipfile.ZipFile(z).extractall("/kaggle/working/m11_seed42_in")
hits = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/verified.parquet", recursive=True) + glob.glob("/kaggle/working/m11_seed42_in/**/verified.parquet", recursive=True)
        if "seed43" not in p and all(os.path.exists(os.path.join(os.path.dirname(p), n)) for n in NEED)]
print("found:", hits)
assert hits, "Attach seed 42's M11 output (m11_full_seed42_outputs.zip) as a dataset"
os.makedirs("reports/m11/full", exist_ok=True)
for n in NEED:
    shutil.copy(os.path.join(hits[0], n), f"reports/m11/full/{n}")
assert json.load(open("reports/m11/full/summary.json"))["seed"] == 42

In [ ]:
# GPU smoke test: load the model, verify 16 made-up rows (one full batch, so memory is tested too). Stops here on failure.
import json, time, torch
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4"
from ids.data.schema import CATEGORIES
from ids.llm.verifier import HFBackend, prompt, verify
be = HFBackend(MODEL, "4bit", "float16", 256)
pay = {"edge_label": "dos", "probs": {c: round(v, 3) for c, v in zip(CATEGORIES, [0.02, 0.35, 0.55, 0.04, 0.02, 0.02])},
       "booster_labels": {"xgboost": "dos", "lightgbm": "ddos"}, "disagreement": 0.31,
       "top_features": [{"feature": "Rate", "value": 2210.0, "shap": 1.4}, {"feature": "Number", "value": 9.5, "shap": 0.6}, {"feature": "Header_Length", "value": 54.0, "shap": -0.3}],
       "class_medians": {"Rate": {"dos": 2500.0, "ddos": 8000.0, "benign": 12.0}, "Number": {"dos": 9.5, "ddos": 9.5, "benign": 5.5}, "Header_Length": {"dos": 54.0, "ddos": 54.0, "benign": 300.0}},
       "recent_class_share": {c: v for c, v in zip(CATEGORIES, [0.03, 0.4, 0.5, 0.07, 0.0, 0.0])}, "drift": {"flag": True, "score": 0.12}, "known_classes": ["benign", "ddos", "dos", "recon"]}
feats = ["Rate", "Number", "Header_Length"]
tic = time.perf_counter()
recs, batches = verify([{"key": i, "payload": pay} for i in range(16)], be, CATEGORIES, feats, batch_size=16, max_retries=2)
sec = time.perf_counter() - tic
print(be.generate([prompt(pay, CATEGORIES)])[0])
print("valid:", sum(r["valid"] for r in recs), "of 16; first try:", sum(r["first_try_valid"] for r in recs), f"; {sec:.1f} s; peak GPU {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")
print(f"rough estimate: {9 * 3600 * 16 / sec:.0f} rows in 9 hours")
del be; torch.cuda.empty_cache()
assert sum(r["valid"] for r in recs) >= 12, "The model's replies mostly fail the contract: stop and bring the output back"

In [ ]:
# Verify until the 9-hour budget runs out, then pack (also after a failure)
!python -X faulthandler scripts/m11_subset.py --env kaggle --set llm.backend=hf llm.model_id={MODEL} --hours 9; echo "exit code: $?"
REP = "reports/m11/full/mistral"
stage = "/kaggle/working/m11_stage"
shutil.rmtree(stage, ignore_errors=True); os.makedirs(stage)
if os.path.isdir(REP):
    shutil.copytree(REP, f"{stage}/{REP}")
for r in sorted(glob.glob("/kaggle/working/results/runs/*_m11_subset_mistral")):
    shutil.copytree(r, f"{stage}/results/runs/{os.path.basename(r)}")
shutil.make_archive("/kaggle/working/m11_mistral_outputs", "zip", stage)
shutil.rmtree(stage)
print("zip written:", sorted(os.listdir(REP)) if os.path.isdir(REP) else [])
if os.path.exists(f"{REP}/VERIFY.md"):
    print(open(f"{REP}/VERIFY.md", encoding="utf-8").read())